In [0]:
import sys
import os
from pathlib import Path

marker = "RETAIL_LAB.json"
cwd = Path(os.getcwd()).resolve()
PROJECT_ROOT = None

print(cwd)

for candidate in [cwd, *cwd.parents]:
    # print(f"Looking for {marker} in {candidate}")
    if (candidate / marker).exists():
        #print("Project root :", candidate)
        PROJECT_ROOT = str(candidate)
        #print("Project root :", PROJECT_ROOT)
        break


if PROJECT_ROOT is None:
    for search_root in [Path("/Workspace/Users"), Path("/Workspace/Shared"), Path("/Workspace")]:
        if not search_root.exists():
            continue
        for current_root, dirs, files in os.walk(search_root):
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in {"Trash", ".Trash"}]
            if marker in files:
                PROJECT_ROOT = current_root
                break
        if PROJECT_ROOT:
            break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project not found. Import the complete ZIP into Databricks Workspace."
    )

SRC_ROOT = f"{PROJECT_ROOT}/src"
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

from bootstrap import ensure_lab

spark.sql("USE CATALOG wsadbnov")

ctx = ensure_lab(
    spark=spark,
    project_root=PROJECT_ROOT,
)

print("Project root :", PROJECT_ROOT)
print("Catalog      :", ctx.catalog)
print("Schema       :", ctx.schema)
print("Volume       :", ctx.volume)
print("Source root  :", ctx.source_root)


In [0]:
location=spark.sql(
    """select current_catalog() as current_catalog , current_schema() as current_schema"""
)

type(location)

curr_catalog=location.first()["current_catalog"]
curr_schema=location.first()["current_schema"]


print(curr_catalog)

if curr_catalog !="wsadbnov":
    raise RuntimeError("Please run the notebook in the correct catalog ")


# location.show()


In [0]:
display(spark.sql("show catalogs"))

In [0]:
display(spark.sql("show schemas"))

In [0]:
display(spark.sql(f"show schemas in {curr_catalog}"))

In [0]:
cat=curr_catalog
display(spark.sql(f"show schemas in `{cat}`"))

In [0]:
%sql
use catalog system

In [0]:
spark.sql(
    """select current_catalog() as current_catalog , current_schema() as current_schema"""
).show()

In [0]:
%sql
use catalog wsadbnov

In [0]:
# create schemas as retail_bronze, retail_silver, retail_gold inside wsadbnov
spark.sql(f"create schema if not exists wsadbnov.retail_bronze")
spark.sql(f"create schema if not exists wsadbnov.retail_silver")
spark.sql(f"create schema if not exists wsadbnov.retail_gold")






In [0]:
%sql
use schema retail_bronze

In [0]:
display(
    spark.sql(f"SHOW VOLUMES IN {curr_catalog}.{"retail_bronze"}"))


In [0]:
display(dbutils.fs.ls("/Volumes/wsadbnov/retail_bronze/raw_files"))

In [0]:
from pyspark.sql import functions as F

orders_csv="/mnt/wsadbnov/retail_bronze/raw_files/orders.csv"
df = spark.read.format("csv").option("header", "true").load("/Workspace/Users/rk3924726@gmail.com/adbnov2026/data/orders.csv")
# display(df)
orders_clean_df=(
    df.withColumn("order_ts", F.to_timestamp(F.col("order_ts"),"yyyy-MM-dd HH:mm:ss"))
    .withColumn("shipping_city", F.upper(F.col("shipping_city")))
    .withColumn("order_status",  F.upper(F.col("order_status")))
    .withColumn("channel",  F.upper(F.col("channel")))
    )
orders_clean_df.show()
# write to


# df.write.mode("overwrite").saveAsTable("wsadbnov.retail_bronze.orders")



In [0]:
orders_clean_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("wsadbnov.retail_bronze.orders")

In [0]:
spark.sql("""
          select * from wsadbnov.retail_bronze.orders
          """).show()

In [0]:
spark.sql(f" describe table extended wsadbnov.retail_bronze.orders ")

In [0]:
display(spark.sql(f" describe table extended wsadbnov.retail_bronze.orders "))

In [0]:
order_tab_info=(spark.sql(f" describe table extended wsadbnov.retail_bronze.orders "))

In [0]:
display(order_tab_info.filter(F.col("col_name").isin("Location")))

In [0]:
order_status_counts=spark.sql(f" select order_status, count(*) as count from wsadbnov.retail_bronze.orders group by order_status ")
display(order_status_counts)


In [0]:
view_name="vw_order"

# select order_status, count(*) as count from wsadbnov.retail_bronze.orders group by order_status

spark.sql(f" CREATE OR REPLACE VIEW wsadbnov.retail_gold.{view_name} as select order_status, count(*) as count from wsadbnov.retail_bronze.orders group by order_status ")


display(spark.sql(f" select * from wsadbnov.retail_gold.{view_name} "))


In [0]:
spark.sql(
    f" show grants on table wsadbnov.retail_gold.{view_name} "
).show()

In [0]:
spark.sql(
    f" show grants on table wsadbnov.retail_bronze.orders "
).show()

In [0]:
spark.sql("""
          DESCRIBE HISTORY wsadbnov.retail_bronze.orders 
          """).show()

In [0]:
spark.sql("""
          select * from wsadbnov.retail_bronze.orders
          """).show()

In [0]:
spark.sql("""
         insert into wsadbnov.retail_bronze.orders values (1,1,1,1,1,1)
          """).show()

In [0]:
spark.sql("""
          select * from wsadbnov.retail_bronze.orders
          """).show(200)

In [0]:
spark.sql("""
          describe history wsadbnov.retail_bronze.orders
          """).show()

In [0]:
spark.sql("""
         insert into wsadbnov.retail_bronze.orders values (2,1,1,1,1,1)
          """).show()

In [0]:
spark.sql("""
          describe history wsadbnov.retail_bronze.orders
          """).show()

In [0]:
spark.sql("""
         delete from wsadbnov.retail_bronze.orders where order_id='2'
          """)

In [0]:
spark.sql("""
          describe history wsadbnov.retail_bronze.orders
          """).show()

In [0]:
spark.sql("""
         update wsadbnov.retail_bronze.orders set customer_id='2' where order_id='1'
          """)

In [0]:
spark.sql("""
          describe history wsadbnov.retail_bronze.orders
          """).show( truncate=False)

In [0]:
try:
    dbutils.fs.ls("abfss://unity-catalog-storage@dbstoragecad66uvapz6yy.dfs.core.windows.net/7405609019767524/__unitystorage/catalogs/dafec870-9c26-474f-9ee4-df79bd13a066")
except Exception as e:
    print(f"Error accessing path: {e}")

In [0]:
spark.sql("""DESCRIBE DETAIL wsadbnov.retail_bronze.orders;""").show()

In [0]:
path = spark.sql(
    "DESCRIBE DETAIL wsadbnov.retail_bronze.orders"
).select("location").collect()[0][0]

df = spark.read.format("delta").load(path)
display(df)

In [0]:
/order/

├── _delta_log/
│   ├── 00000000000000000000.json
│   ├── 00000000000000000001.json
│   ├── 00000000000000000010.checkpoint.parquet
│   └── ...
│
├── part-00000-xxxx.snappy.parquet   -actual data
├── part-00001-xxxx.snappy.parquet
├── part-00002-xxxx.snappy.parquet
└── ...


/customer/

├── _delta_log/
│   ├── 00000000000000000000.json
│   ├── 00000000000000000001.json
│   ├── 00000000000000000010.checkpoint.parquet
│   └── ...
│
├── part-00000-xxxx.snappy.parquet
├── part-00001-xxxx.snappy.parquet
├── part-00002-xxxx.snappy.parquet
└── ...


In [0]:
order_df.write.format("parquet").mode("overwrite").save("/tmp/delta/orders")

In [0]:
spark.sql("""
          describe history wsadbnov.retail_bronze.orders
          """).show( truncate=False)

In [0]:
spark.sql("""
          select * from wsadbnov.retail_bronze.orders as of timestamp
          """).show( 200)

In [0]:
spark.sql("""
          create table wsadbnov.retail_silver.orders_4 as select * from wsadbnov.retail_bronze.orders version as of 4
          """)

In [0]:
spark.sql("""
          describe history wsadbnov.retail_silver.orders_4 
          """).show()

In [0]:
MAX
MIN
NULL
COUNT   


part-00000-xxxx.snappy.parquet -- MAX,MIN,NULL,COUNT
├── part-00001-xxxx.snappy.parquet -- MAX,MIN,NULL,COUNT
├── part-00002-xxxx.snappy.parquet- -- MAX,MIN,NULL,COUNT


part-00000-xxxx.snappy.parquet -- order id  from 0-100   100 , 0 0 100
part-00001-xxxx.snappy.parquet -- order id  from 100-200 200  100 0 100

part-00002-xxxx.snappy.parquet -- order id  from 200-300 300  200 0 100


select * from orders where order_id=250

1000GB -- files ---> file.zip -- GB




